Part 1 MTH 348 Homework

In [5]:
# imports
import math
import timeit
import numpy as np
import pandas as pd
import scipy.linalg
from sklearn import metrics

In [6]:
def regr_metrics(act, pred):
    return (math.sqrt(metrics.mean_squared_error(act, pred)), 
     metrics.mean_absolute_error(act, pred))

In [7]:
def scipylstq(A, b):
    return scipy.linalg.lstsq(A,b)[0]

In [8]:
row_names = ['Normal Eqns- Naive',
             'Normal Eqns- Cholesky', 
             'QR Factorization', 
             'SVD', 
             'Scipy lstsq']

name2func = {'Normal Eqns- Naive': 'ls_naive', 
             'Normal Eqns- Cholesky': 'ls_chol', 
             'QR Factorization': 'ls_qr',
             'SVD': 'ls_svd',
             'Scipy lstsq': 'scipylstq'}

In [9]:
def ls_naive(A, b):
     return np.linalg.inv(A.T @ A) @ A.T @ b

def ls_chol(A, b):
    R = scipy.linalg.cholesky(A.T @ A)
    w = scipy.linalg.solve_triangular(R, A.T @ b, trans='T')
    return scipy.linalg.solve_triangular(R, w)

def ls_qr(A,b):
    Q, R = scipy.linalg.qr(A, mode='economic')
    return scipy.linalg.solve_triangular(R, Q.T @ b)

def ls_svd(A,b):
    m, n = A.shape
    U, sigma, Vh = scipy.linalg.svd(A, full_matrices=False, lapack_driver='gesdd')
    w = (U.T @ b)/ sigma
    return Vh.T @ w

In [10]:
m_array = np.array([100, 1000, 10000])
n_array = np.array([20, 100, 1000])

In [11]:
index = pd.MultiIndex.from_product([m_array, n_array], names=['# rows', '# cols'])

In [12]:
pd.options.display.float_format = '{:,.6f}'.format
df = pd.DataFrame(index=row_names, columns=index)
df_error = pd.DataFrame(index=row_names, columns=index)

In [13]:
# %%prun
for m in m_array:
    for n in n_array:
        if m >= n:        
            x = np.random.uniform(-10,10,n)
            A = np.random.uniform(-40,40,[m,n])   # removed np.asfortranarray
            b = np.matmul(A, x) + np.random.normal(0,2,m)
            for name in row_names:
                fcn = name2func[name]
                t = timeit.timeit(fcn + '(A,b)', number=5, globals=globals())
                df.loc[name, [(m,n)]] = t
                coeffs = locals()[fcn](A, b)
                reg_met = regr_metrics(b, A @ coeffs)
                df_error.at[name, (m,n)] = reg_met[0]

In [14]:
df

# rows                   100                    1000                     \
# cols                    20       100  1000     20       100      1000   
Normal Eqns- Naive    0.000375 0.110252  NaN 0.000476 0.021522 0.386641   
Normal Eqns- Cholesky 0.001340 0.001532  NaN 0.000814 0.002536 0.106732   
QR Factorization      0.000739 0.003795  NaN 0.003407 0.046595 2.069530   
SVD                   0.001596 0.015998  NaN 0.004086 0.104030 1.985716   
Scipy lstsq           0.001555 0.008355  NaN 0.002632 0.033955 0.854768   

# rows                   10000                     
# cols                    20       100       1000  
Normal Eqns- Naive    0.004874 0.042582  1.527154  
Normal Eqns- Cholesky 0.001957 0.017692  0.386083  
QR Factorization      0.014763 0.241099  5.671994  
SVD                   0.021192 0.285685 11.559289  
Scipy lstsq           0.012164 0.168656  3.357031